In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- CELL 1: DEFINE FUNCTIONS ---
def get_file_path(filename):
    """Checks for the file in the current folder or a 'data' folder."""
    if os.path.exists(filename):
        return filename
    elif os.path.exists(os.path.join('data', filename)):
        return os.path.join('data', filename)
    else:
        print(f"Warning: {filename} not found.")
        return None

def clean_nrb_df(df):
    """Cleans the NRB CSV format into a usable time-series DataFrame."""
    # Rename first column and transpose
    df = df.rename(columns={df.columns[0]: 'Indicator'})
    df = df.set_index('Indicator').transpose()
    
    # Extract year (e.g., '2023/24R' becomes 2023)
    df.index = df.index.str.extract('(\d{4})')[0].astype(int)
    
    # Convert all values to numbers (removes '...' and '-')
    df = df.apply(pd.to_numeric, errors='coerce')
    return df

# --- CELL 2: LOAD AND MERGE DATA ---
file_2024_name = 'Macroeconomic-Indicators-of-Nepal-2024-December.xlsx - 3.csv'
file_2025_name = 'Macroeconomic-Indicators-of-Nepal-2025-December.xlsx - 3.csv'

path_24 = get_file_path(file_2024_name)
path_25 = get_file_path(file_2025_name)

if path_24 and path_25:
    df_2024 = pd.read_csv(path_24, skiprows=4)
    df_2025 = pd.read_csv(path_25, skiprows=4)

    df_24_clean = clean_nrb_df(df_2024)
    df_25_clean = clean_nrb_df(df_2025)

    # Merge: Prioritize 2025 data, fill historical gaps with 2024
    df_final = df_25_clean.combine_first(df_24_clean).sort_index()
    print("Success: Data loaded and merged.")
else:
    print("Error: Could not find the necessary CSV files. Check your folder.")

# --- CELL 3: ANALYSIS & PLOTTING ---
if 'df_final' in locals():
    # Use exact column names from Table 3
    cols = ["Real GDP at Purchasers' Price", "Consumer Price Index (Annual Average)", "Total Revenue"]
    
    # Create analysis dataframe
    df_analysis = df_final[cols].copy()
    df_analysis['Revenue_Growth'] = df_analysis['Total Revenue'].pct_change() * 100

    # Plotting
    plt.figure(figsize=(10, 5))
    plt.plot(df_analysis.index, df_analysis["Real GDP at Purchasers' Price"], marker='o', label='GDP Growth %')
    plt.axhline(y=3, color='r', linestyle='--', label='Slowdown Threshold')
    plt.title("Nepal Economic Growth Trend")
    plt.legend()
    plt.show()

Error: Could not find the necessary CSV files. Check your folder.
